### Ingestión de la carpeta "production_company"

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_environment","")
v_environment = dbutils.widgets.get("p_environment")

###Paso 1 - Leer los archivos CSV usando "DataFrameReader" de Spark
####La documentación para los comandos de Spark la sacamos de https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/index.html

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

production_companies_schema = StructType(fields = [
    StructField("companyId", IntegerType(), True),
    StructField("companyName", StringType(), True)
])


In [0]:
#production_companies_df = spark.read.schema(production_companies_schema).csv("abfss://bronze@moviee.dfs.core.windows.net/production_company")

production_companies_df = spark.read.schema(production_companies_schema).csv(f"{bronze_folder_path}/production_company")






In [0]:
production_companies_df.printSchema()

In [0]:
display(production_companies_df)

In [0]:
production_companies_df.count()


### Paso 2 - Renombrar las columnas y añadir columnas nuevas

In [0]:
from pyspark.sql.functions import current_timestamp, lit

#production_companies_final_df = production_companies_df \
#                        .withColumnsRenamed({"companyId": "company_id",
#                                             "companyName": "company_name"}) \
#                        .withColumn("ingestion_date", current_timestamp()) \
#                        .withColumn("environment", lit("production"))

                        
production_companies_final_df = production_companies_df \
                               .withColumnsRenamed({"companyId": "company_id",
                                               "companyName": "company_name"})

production_companies_final_df = add_ingestion_date(production_companies_final_df) \
                                .withColumn("environment", lit(v_environment))

In [0]:
display(production_companies_final_df)

### Paso 3 - Escribir la salida en un formato "Parquet"

In [0]:
#production_companies_final_df.write.mode("overwrite").parquet("abfss://silver@moviee.dfs.core.windows.net/production_companies")

production_companies_final_df.write.mode("overwrite").parquet(f"{silver_folder_path}/production_companies")

In [0]:
#display(spark.read.parquet("abfss://silver@moviee.dfs.core.windows.net/production_companies"))

display(spark.read.parquet(f"{silver_folder_path}/production_companies"))

### Paso 4 - Escribir datos en una managed table en el contenedor silver

#### Esto es un cambio posterior en el notebook, pasaría a reemplazar al paso 3, ya que ahora no quiero más generar archivos de salida en la capa silver a partir del dataframe, sino meter esa info en una tabla administrada por spark, pero dejo igual lo del paso 3, ya que esto se crea dentro de la carpeta _unitystorage y no pisa lo del paso 3

In [0]:
production_companies_final_df.write.mode("overwrite").format("delta").saveAsTable("movie_silver.production_companies")

In [0]:
%sql
SELECT * FROM movie_silver.production_companies

In [0]:
dbutils.notebook.exit("Success")